In [10]:
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# ==========================================
# 1. Custom Web Scraper Function
# Replacing the external 'scraper' module
# ==========================================

def fetch_website_contents(url):
    """
    Fetches the HTML content of a URL, extracts the text, 
    and cleans it up for the LLM.
    """
    try:
        # Add headers to simulate a real browser request
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # Parse HTML and extract text
        soup = BeautifulSoup(response.content, 'html.parser')
        text = soup.get_text(separator=' ', strip=True)
        
        # Limit text length to avoid exceeding Phi-3's context window
        return text[:4000] 
        
    except Exception as e:
        return f"Failed to fetch content. Error: {e}"

# ==========================================
# 2. Summarization Pipeline
# URL -> Scraping -> Content -> Phi-3 -> Summary
# ==========================================

def run_summarization_pipeline(url):
    print(f"Fetching website content from: {url} ...")
    website_content = fetch_website_contents(url)
    
    # Check if scraping failed before sending to the model
    if website_content.startswith("Failed to fetch"):
        print(website_content)
        return
        
    # Define the system behavior and user instructions
    system_prompt = """
    You are an assistant that analyzes the contents of a website,
    and provides a short summary of this website, ignoring text that might be navigation related.
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
    """
    
    user_prompt_prefix = """
    Here are the contents of a website.
    Provide a short summary of this website.
    If it includes news or announcements, then summarize these too.

    """
    
    # Combine the system prompt, user prompt, and the scraped content
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_content}
    ]
    
    # Initialize the local OpenAI client pointing to the Ollama instance
    print("Sending content to the local Phi-3 model...")
    OLLAMA_BASE_URL = "http://localhost:11434/v1"
    
    # Using 'ollama' as a dummy API key for the local instance
    ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama') 
    
    # Generate the completion using the phi3 model
    response = ollama_client.chat.completions.create(
        model="phi3",
        messages=messages,
        temperature=0.3 # Low temperature for more factual summaries
    )
    
    # Extract the text from the response and render it as Markdown
    summary = response.choices[0].message.content
    print("\n--- Summary Generated ---\n")
    display(Markdown(summary))

# ==========================================
# 3. Pipeline Execution
# ==========================================

# Sample URL for testing (A Wikipedia article about Large Language Models)
target_url = "https://en.wikipedia.org/wiki/Large_language_model"

# Run the pipeline
run_summarization_pipeline(target_url)

Fetching website content from: https://en.wikipedia.org/wiki/Large_language_model ...
Failed to fetch content. Error: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Max retries exceeded with url: /wiki/Large_language_model (Caused by NewConnectionError("HTTPSConnection(host='en.wikipedia.org', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable"))
